# Using Llama with Python

This project is a practical introduction to running **Meta Llama models locally from Python** using **Ollama**.

## What you will learn

- Install and verify the Python dependencies
- Connect Python to a local Llama model through Ollama
- Send prompts and receive responses
- Control generation parameters
- Build a multi-turn conversation
- Request structured JSON output
- Stream Llama responses token-by-token
- Build a small command-line style chatbot

The approach in this notebook keeps the model local. You do not need an OpenAI API key or a paid cloud API.

## 1. Prerequisites

Install **Ollama** on your computer and make sure the Ollama service is running.

Then download a Llama model from a terminal. For example:

```bash
ollama pull llama3.2
```

You can verify the model with:

```bash
ollama list
```

This notebook communicates with Ollama through its local HTTP API.

In [1]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Miguel Estrada\AppData\Local\Programs\Python\Python37\python.exe -m pip install --upgrade pip' command.


In [2]:
import requests
import json
from pprint import pprint

OLLAMA_URL = "http://localhost:11434"
MODEL = "llama3.2"

print(f"Ollama URL: {OLLAMA_URL}")
print(f"Model: {MODEL}")

Ollama URL: http://localhost:11434
Model: llama3.2


## 2. Test the Ollama connection

Ollama exposes a local REST API. We can use Python's `requests` library to communicate with it.

In [3]:
try:
    response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    response.raise_for_status()
    data = response.json()

    print("Ollama is running!\n")
    print("Available models:")
    for model in data.get("models", []):
        print("-", model.get("name"))
except requests.exceptions.RequestException as error:
    print("Could not connect to Ollama.")
    print("Make sure Ollama is installed and running.")
    print("Error:", error)

Ollama is running!

Available models:
- llama3.2:latest


## 3. Create a reusable Llama function

The `/api/generate` endpoint accepts a prompt and returns a generated response.

In [4]:
def ask_llama(prompt, model=MODEL, temperature=0.7):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json=payload,
        timeout=120
    )
    response.raise_for_status()
    return response.json()["response"]

## 4. Your first Llama prompt

In [5]:
prompt = "Explain what artificial intelligence is in three simple sentences."

answer = ask_llama(prompt)
print(answer)

Artificial intelligence (AI) refers to the simulation of human intelligence in machines, allowing them to learn, reason, and interact with the environment. AI systems use algorithms and data to make decisions, classify patterns, and solve problems, often with the ability to improve over time through machine learning. By mimicking the cognitive abilities of humans, AI has the potential to automate tasks, enhance decision-making, and revolutionize various industries.


## 5. Prompt engineering

A useful prompt usually provides:

1. A role or context
2. A clear task
3. Constraints
4. The desired output format

For example, we can ask Llama to behave like a Python tutor.

In [6]:
prompt = """
You are a Python programming tutor.

Explain the difference between a list and a tuple.
The student is a beginner.
Use one short example for each.
"""

print(ask_llama(prompt))

Welcome to Python programming! I'm excited to help you learn.

In Python, lists and tuples are both data structures that can store multiple values. However, there's a key difference between them:

**Lists are Mutable**: A list is a collection of values that can be modified after creation. You can add, remove, or modify elements in a list.

**Tuples are Immutable**: A tuple is a collection of values that cannot be modified after creation. You cannot add, remove, or modify elements in a tuple.

Let's see an example of each:

**List Example:**
```python
my_list = [1, 2, 3, 4, 5]
print(my_list)

# Output: [1, 2, 3, 4, 5]

# Modifying the list
my_list.append(6)
print(my_list)

# Output: [1, 2, 3, 4, 5, 6]
```
In this example, we create a list `my_list` and print its initial values. We then use the `append()` method to add a new value to the list, demonstrating that lists are mutable.

**Tuple Example:**
```python
my_tuple = (1, 2, 3, 4, 5)
print(my_tuple)

# Output: (1, 2, 3, 4, 5)

# Tryin

## 6. Generation parameters

The `temperature` parameter controls how deterministic the generated answer tends to be.

- Lower values: more predictable responses
- Higher values: more varied and creative responses

For code generation, a lower temperature is often useful. For brainstorming, a higher value can be useful.

In [7]:
prompt = "Give me one creative name for a Python AI project."

print("Low temperature:")
print(ask_llama(prompt, temperature=0.2))

print("\nHigher temperature:")
print(ask_llama(prompt, temperature=1.2))

Low temperature:
Here's a creative name for a Python AI project:

"EchoMind"

This name suggests the idea of a mind that echoes or reflects the thoughts and ideas of others, while also hinting at the AI's ability to learn and adapt. It's catchy and easy to remember, making it a great name for a Python AI project.

Higher temperature:
How about "Luminari"? It suggests intelligence, insight, and illumination, which are all fitting themes for an AI project.


## 7. Multi-turn conversations

The simplest way to maintain context is to keep the previous messages and send them as part of the next request.

Ollama also provides a `/api/chat` endpoint specifically designed for conversations.

In [8]:
def chat_with_llama(messages, model=MODEL, temperature=0.7):
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }

    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json=payload,
        timeout=120
    )
    response.raise_for_status()
    return response.json()["message"]["content"]

In [10]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful Python tutor. Keep explanations concise."
    },
    {
        "role": "user",
        "content": "What is a Python dictionary?"
    }
]

answer = chat_with_llama(messages)
print(answer)

messages.append({"role": "assistant", "content": answer})
messages.append({
    "role": "user",
    "content": "Show me a simple example."
})

print(chat_with_llama(messages))

**Python Dictionary**

A Python dictionary is an unordered collection of key-value pairs. It is a mutable data structure that stores data in a way that allows for efficient lookups, insertions, and deletions.

**Basic Syntax**

A dictionary is defined using the `{}` syntax, where each key-value pair is separated by a comma. For example:
```python
my_dict = {'name': 'John', 'age': 30, 'city': 'New York'}
```
In this example, `name`, `age`, and `city` are the keys, and `'John'`, `30`, and `'New York'` are the corresponding values.

**Accessing Values**

You can access a value in a dictionary by its key using the square bracket notation. For example:
```python
print(my_dict['name'])  # Output: John
```
**Example Use Cases**

Dictionaries are useful when you need to store and retrieve data with a unique identifier (key). Here are a few examples:

* Storing user data with a unique username or ID
* Caching data with a unique key-value pair
* Representing a configuration file with key-value p

## 8. Structured JSON output

For applications, structured output is often more useful than plain text. We can ask Llama to return JSON and then parse it with Python.

In [11]:
def ask_llama_json(prompt, model=MODEL):
    payload = {
        "model": model,
        "prompt": prompt,
        "format": "json",
        "stream": False
    }

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json=payload,
        timeout=120
    )
    response.raise_for_status()
    return response.json()["response"]


prompt = """
Create information about a fictional Python course.
Return an object with these fields:
- title
- difficulty
- duration_hours
- topics
"""

raw_json = ask_llama_json(prompt)
course = json.loads(raw_json)
pprint(course)

{'difficulty': 'Beginner',
 'duration_hours': 40,
 'title': 'Python Fundamentals',
 'topics': ['Introduction to Python',
            'Data Types and Operators',
            'Control Structures',
            'Functions',
            'Modules and Packages',
            'File Input/Output',
            'Object-Oriented Programming',
            'Data Structures',
            'Web Development with Flask']}


## 9. Streaming responses

Instead of waiting for the complete answer, we can stream the response as Llama generates it. This is useful for chat interfaces.

In [12]:
def stream_llama(prompt, model=MODEL):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": True
    }

    with requests.post(
        f"{OLLAMA_URL}/api/generate",
        json=payload,
        stream=True,
        timeout=120
    ) as response:
        response.raise_for_status()

        for line in response.iter_lines():
            if line:
                data = json.loads(line)
                print(data.get("response", ""), end="", flush=True)


stream_llama("Explain how a neural network works for a beginner.")
print()

I'd be happy to explain how a neural network works in simple terms.

**What is a Neural Network?**

A neural network is a computer system that is inspired by the way our brains work. Just like our brains, a neural network is made up of many interconnected nodes or "neurons" that process and transmit information.

**How Does a Neural Network Work?**

Here's a simplified overview of how a neural network works:

1. **Input**: The neural network takes in data, such as images, text, or audio. This data is fed into the network through one of its many inputs.
2. **Layer 1: Perception**: The first layer of the network, called the input layer, receives the input data. This layer is made up of many nodes (neurons) that apply different weights to the input data to adjust its strength.
3. **Layer 2: Processing**: The output from the input layer is passed on to the next layer, called the hidden layer. This layer is made up of many more nodes (neurons) that process the data in a complex way. Each no

## 10. Build a simple chatbot

The following function maintains the conversation history and sends each new user message to Llama.

In [13]:
def run_chatbot():
    messages = [
        {
            "role": "system",
            "content": "You are a friendly Python programming assistant."
        }
    ]

    print("Llama chatbot started. Type 'exit' to stop.\n")

    while True:
        user_input = input("You: ")

        if user_input.lower().strip() == "exit":
            print("Goodbye!")
            break

        messages.append({"role": "user", "content": user_input})

        try:
            answer = chat_with_llama(messages)
            messages.append({"role": "assistant", "content": answer})
            print(f"Llama: {answer}\n")
        except requests.exceptions.RequestException as error:
            print("Error communicating with Ollama:", error)
            messages.pop()

# Uncomment the next line to start the chatbot.
# run_chatbot()

## 11. Mini project: AI programming assistant

Let's create a small reusable assistant that can:

- Explain code
- Find possible bugs
- Suggest improvements
- Generate documentation

This pattern can later become the backend of a web application, REST API, React application, or MCP-based application.

In [ ]:
def code_assistant(code, task="explain"):
    instructions = {
        "explain": "Explain what this code does for a beginner.",
        "debug": "Analyze the code for bugs and explain how to fix them.",
        "improve": "Suggest practical improvements to this code.",
        "document": "Generate concise documentation for this code."
    }

    if task not in instructions:
        raise ValueError(f"Unknown task: {task}")

    prompt = f"""
        You are an expert Python developer.

        {instructions[task]}

        Code:
        ```python
        {code}
        ```
        """

    return ask_llama(prompt, temperature=0.3)


example_code = """
def calculate_average(numbers):
    return sum(numbers) / len(numbers)
"""

print(code_assistant(example_code, "explain"))

**Welcome to Python Programming!**

I'm excited to explain this simple yet powerful code to you.

**What does this code do?**

This code defines a function called `calculate_average` that takes a list of numbers as input and returns their average value.

**Let's break it down:**

1. `def calculate_average(numbers):` This line defines a new function called `calculate_average`. The word `def` is short for "define", and it's used to create a new function.
2. `return sum(numbers) / len(numbers)` This line is the heart of the function. It calculates the average of the input numbers.

Here's what's happening here:

* `sum(numbers)` adds up all the numbers in the list.
* `len(numbers)` counts the total number of elements in the list.
* The result of `sum(numbers)` is then divided by the result of `len(numbers)`, which gives us the average value.

**Example:**

Suppose we call the function like this:
```python
numbers = [1, 2, 3, 4, 5]
average = calculate_average(numbers)
print(average)  # Out

## Architecture

```text
Jupyter / Python
       |
       | HTTP
       v
   Ollama API
       |
       v
   Llama model
       |
       v
 Generated response
```

This architecture is a good starting point for building a local AI application without sending prompts to a hosted API.